# Quantum Resource Estimation with ORMAS-CI

This notebook demonstrates genuine quantum computing resource estimation
using ORMAS-CI and QDK/Chemistry v1.0.2. All metrics are real QDK-computed
values -- no analytical estimates or proxy metrics.

Sections:
1. **Qubit Hamiltonian Construction** -- Jordan-Wigner mapping via QDK
2. **Classical Qubit Solve** -- energy verification
3. **Real Circuit Construction** -- gate counts from QASM parsing
4. **Benchmark Comparison** -- CASCI vs RASCI vs ORMAS on real systems
5. **Scaling Analysis** -- real Pauli term counts across system sizes

In [ ]:
import numpy as np
from pyscf import gto, scf, mcscf
from ormas_ci import ORMASFCISolver, ORMASConfig, Subspace
from ormas_ci.determinants import count_determinants, casci_determinant_count
from ormas_ci.subspaces import RASConfig

try:
    from qdk_chemistry.algorithms import (
        QdkQubitMapper, QdkHamiltonianConstructor, create,
    )
    from qdk_chemistry.algorithms.time_evolution.builder.trotter import Trotter
    from qdk_chemistry.algorithms.time_evolution.controlled_circuit_mapper import PauliSequenceMapper
    from qdk_chemistry.data.time_evolution.controlled_time_evolution import ControlledTimeEvolutionUnitary
    from qdk_chemistry.data import Structure
    from qdk_chemistry.plugins.pyscf.scf_solver import PyscfScfSolver
    QDK_AVAILABLE = True
    print('qdk-chemistry available')
except ImportError:
    QDK_AVAILABLE = False
    print('qdk-chemistry not installed. Install with: pip install qdk-chemistry')

## 1. Qubit Hamiltonian Construction

Under Jordan-Wigner mapping, each spatial orbital maps to 2 qubits.
The second-quantized Hamiltonian becomes a sum of weighted Pauli strings.
We use QDK to build **real** qubit Hamiltonians, not estimates.

In [ ]:
def build_qubit_ham(xyz_str, basis, n_active_elec, n_active_orb):
    """Build active-space qubit Hamiltonian via QDK."""
    structure = Structure.from_xyz(xyz_str)
    _, wfn = PyscfScfSolver().run(structure, 0, 1, basis)
    as_sel = create('active_space_selector', 'qdk_valence',
                    num_active_electrons=n_active_elec,
                    num_active_orbitals=n_active_orb)
    sel_wfn = as_sel.run(wfn)
    sel_orbs = sel_wfn.get_orbitals()
    ham = QdkHamiltonianConstructor().run(sel_orbs)
    qh = QdkQubitMapper().run(ham)
    return qh, ham

if QDK_AVAILABLE:
    # Formaldehyde CAS(4,3)/cc-pVDZ -- n/pi/pi* system
    xyz_ch2o = '4\nCH2O\nC 0 0 0\nO 0 0 1.208\nH 0 0.943 -0.561\nH 0 -0.943 -0.561'
    qh, ham = build_qubit_ham(xyz_ch2o, 'cc-pVDZ', 4, 3)

    print('Formaldehyde CAS(4,3)/cc-pVDZ Qubit Hamiltonian')
    print('=' * 50)
    print(f'Qubits:        {qh.num_qubits}')
    print(f'Pauli terms:   {len(qh.pauli_strings)}')
    print(f'Encoding:      {qh.encoding}')
    print(f'Core energy:   {ham.get_core_energy():.8f} Ha')
    print()
    print('First 10 Pauli strings:')
    for ps, c in list(zip(qh.pauli_strings, qh.coefficients))[:10]:
        print(f'  {c.real:+.8f} * {ps}')
    if len(qh.pauli_strings) > 10:
        print(f'  ... ({len(qh.pauli_strings) - 10} more terms)')
else:
    print('QDK not available.')

## 2. Classical Qubit Solve

Verify that exact diagonalization of the qubit Hamiltonian gives the
same energy as PySCF CASCI.

In [ ]:
if QDK_AVAILABLE:
    solver = create('qubit_hamiltonian_solver', 'qdk_sparse_matrix_solver')
    e_qubit_raw, _ = solver.run(qh)
    e_qubit = e_qubit_raw + ham.get_core_energy()

    # PySCF CASCI for comparison
    mol = gto.M(atom='C 0 0 0; O 0 0 1.208; H 0 0.943 -0.561; H 0 -0.943 -0.561',
                basis='cc-pVDZ', verbose=0)
    mf = scf.RHF(mol); mf.verbose = 0; mf.run()
    mc = mcscf.CASCI(mf, 3, 4); mc.verbose = 0
    e_casci = mc.kernel()[0]

    print('Energy Comparison')
    print('=' * 50)
    print(f'QDK qubit solve:  {e_qubit:.10f} Ha')
    print(f'PySCF CASCI:      {e_casci:.10f} Ha')
    print(f'Qubit dimension:  2^{qh.num_qubits} = {2**qh.num_qubits}')
    print(f'CASCI n_det:      {casci_determinant_count(3, (2,2))}')

## 3. Real Circuit Construction

Gate counts from actual QDK Trotter circuit construction (QASM parsing),
not analytical formulas.

In [ ]:
def get_real_gate_counts(qh):
    """Build real Trotter circuit and parse QASM for gate counts."""
    trotter = Trotter()
    evolution = trotter.run(qh, 1.0)
    n_q = evolution.get_num_qubits()
    ctrl_evo = ControlledTimeEvolutionUnitary(evolution, [n_q])
    circuit = PauliSequenceMapper().run(ctrl_evo)
    # Parse gates inside gate definitions
    counts = {}
    in_def = False
    for line in circuit.qasm.split('\n'):
        s = line.strip().rstrip(';')
        if s.startswith('gate '): in_def = True; continue
        if s == '}': in_def = False; continue
        if not in_def or not s or s == '{': continue
        g = s.split('(')[0].split(' ')[0]
        if g: counts[g] = counts.get(g, 0) + 1
    return counts

if QDK_AVAILABLE:
    gc = get_real_gate_counts(qh)
    print(f'Formaldehyde CAS(4,3)/cc-pVDZ -- 1 Trotter step circuit:')
    print(f'  Total gates: {sum(gc.values())}')
    for g, c in sorted(gc.items(), key=lambda x: -x[1]):
        print(f'    {g:>5}: {c}')
    print()
    print('These are real gate counts from QDK circuit construction,')
    print('not analytical estimates.')

## 4. Benchmark Comparison: CASCI vs RASCI vs ORMAS

Compare methods on formaldehyde (n/pi/pi* system) showing energy accuracy,
determinant reduction, and quantum resource metrics.

In [ ]:
mol = gto.M(atom='C 0 0 0; O 0 0 1.208; H 0 0.943 -0.561; H 0 -0.943 -0.561',
            basis='cc-pVDZ', verbose=0)
mf = scf.RHF(mol); mf.verbose = 0; mf.run()
ncas, nelecas = 3, (2, 2)

# CASCI reference
mc = mcscf.CASCI(mf, ncas, sum(nelecas)); mc.verbose = 0
e_ref = mc.kernel()[0]
n_det_full = casci_determinant_count(ncas, nelecas)

# RAS: n_O / pi / pi*
ras = RASConfig(ras1_orbitals=[0], ras2_orbitals=[1], ras3_orbitals=[2],
                max_holes_ras1=1, max_particles_ras3=1, nelecas=nelecas)
mc_ras = mcscf.CASCI(mf, ncas, sum(nelecas)); mc_ras.verbose = 0
mc_ras.fcisolver = ORMASFCISolver(ras.to_ormas_config())
e_ras = mc_ras.kernel()[0]
n_det_ras = count_determinants(ras.to_ormas_config())

# ORMAS: [n_O] / [pi, pi*]
ormas_cfg = ORMASConfig(
    subspaces=[Subspace('n_O', [0], 1, 2), Subspace('pi', [1, 2], 1, 3)],
    n_active_orbitals=ncas, nelecas=nelecas)
mc_ormas = mcscf.CASCI(mf, ncas, sum(nelecas)); mc_ormas.verbose = 0
mc_ormas.fcisolver = ORMASFCISolver(ormas_cfg)
e_ormas = mc_ormas.kernel()[0]
n_det_ormas = count_determinants(ormas_cfg)

print('Formaldehyde CAS(4,3)/cc-pVDZ: CASCI vs RASCI vs ORMAS')
print('=' * 65)
print(f'{"Method":<10} {"Energy(Ha)":>16} {"dE(mHa)":>9} {"n_det":>6} {"Reduction":>10}')
print('-' * 65)
print(f'{"CASCI":<10} {e_ref:>16.8f} {"-":>9} {n_det_full:>6} {"-":>10}')
print(f'{"RASCI":<10} {e_ras:>16.8f} {(e_ras-e_ref)*1000:>+9.3f} {n_det_ras:>6} {n_det_full/n_det_ras:>9.1f}x')
print(f'{"ORMAS":<10} {e_ormas:>16.8f} {(e_ormas-e_ref)*1000:>+9.3f} {n_det_ormas:>6} {n_det_full/n_det_ormas:>9.1f}x')

if QDK_AVAILABLE:
    print(f'\nQubit Hamiltonian: {qh.num_qubits} qubits, {len(qh.pauli_strings)} Pauli terms')
    gc = get_real_gate_counts(qh)
    print(f'1-Trotter-step circuit: {gc.get("cx", 0)} CNOTs, {sum(gc.values())} total gates')

## 5. Scaling: Real Pauli Term Counts

Build actual qubit Hamiltonians for molecules of increasing size and
report real Pauli term counts and CNOT counts. No analytical formulas.

In [ ]:
if QDK_AVAILABLE:
    systems = [
        ('H2/STO-3G', '2\nH2\nH 0 0 0\nH 0 0 0.74', 'sto-3g', 2, 2),
        ('CH2O/cc-pVDZ', '4\nCH2O\nC 0 0 0\nO 0 0 1.208\nH 0 0.943 -0.561\nH 0 -0.943 -0.561',
         'cc-pVDZ', 4, 3),
        ('C2H4/cc-pVDZ', '6\nC2H4\nC 0 0 0\nC 0 0 1.339\nH 0 0.929 -0.561\nH 0 -0.929 -0.561\nH 0 0.929 1.9\nH 0 -0.929 1.9',
         'cc-pVDZ', 2, 2),
        ('C2H4/cc-pVTZ', '6\nC2H4\nC 0 0 0\nC 0 0 1.339\nH 0 0.929 -0.561\nH 0 -0.929 -0.561\nH 0 0.929 1.9\nH 0 -0.929 1.9',
         'cc-pVTZ', 4, 4),
        ('O3/cc-pVDZ', '3\nO3\nO 0 0 0\nO 0 1.278 0\nO 1.107 0.639 0',
         'cc-pVDZ', 6, 5),
    ]

    print('Real Pauli Term Counts and Circuit Metrics')
    print('=' * 80)
    print(f'{"System":<18} {"qubits":>6} {"pauli":>7} {"CNOTs":>7} {"total_gates":>11}')
    print('-' * 80)

    for name, xyz, basis, n_elec, n_orb in systems:
        qh_s, _ = build_qubit_ham(xyz, basis, n_elec, n_orb)
        gc_s = get_real_gate_counts(qh_s)
        print(f'{name:<18} {qh_s.num_qubits:>6} {len(qh_s.pauli_strings):>7} '
              f'{gc_s.get("cx", 0):>7} {sum(gc_s.values()):>11}')

    print()
    print('All values computed from actual QDK Jordan-Wigner mapping')
    print('and Trotter circuit construction.')
else:
    print('QDK not available. Run with qdk-chemistry installed for real metrics.')

## Summary

All metrics in this notebook are **real computed values** from QDK/Chemistry:

- **Pauli term counts**: from `QdkQubitMapper.run()`, not formulas
- **Gate counts**: from QASM parsing of `PauliSequenceMapper` circuits, not estimates
- **Energies**: from `SparseMatrixSolver` exact diagonalization
- **Determinant counts**: from `build_determinant_list()` / `casci_determinant_count()`

Run the full benchmark suite with:
```bash
python benchmarks/bench_qdk_quantum.py --all
```